# Truck Blind Spot Detection - Notebook Demo

Notebook này chạy pipeline đúng với project hiện tại: YOLOv9 detection, multi-zone ROI, Kalman tracking, motion prediction và visualizer gọn cho demo. Mặc định notebook dùng cùng cấu hình với `app.py`:

| Thành phần | Giá trị |
|---|---|
| Weights | `weights/best_roiv2.pt` |
| Video demo | `assets/videos/demo4.mp4` |
| ROI config | `configs/roi.json` |
| ROI profile | `front_camera` |
| Class config | `configs/classes.yaml` |

Notebook chạy được trên máy local hoặc Google Colab. Trên Colab, bật GPU trong **Runtime -> Change runtime type** trước khi chạy các cell.


## 0. Chuẩn bị môi trường

Cell này tự nhận diện repo hiện tại. Nếu đang ở Colab và chưa có repo, cell sẽ clone project từ GitHub vào `/content/truck_blind_spot`. Mặc định chỉ cài `requirements.txt` khi chạy trong Colab để tránh thay đổi môi trường local ngoài ý muốn.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/VTD0102/truck_blind_spot.git"
COLAB_PROJECT_ROOT = Path("/content/truck_blind_spot")

try:
    import google.colab  # type: ignore[import-not-found]

    IS_COLAB = True
except ImportError:
    IS_COLAB = False

current_dir = Path.cwd().resolve()
if not (current_dir / "src" / "pipeline.py").exists():
    if IS_COLAB:
        if not COLAB_PROJECT_ROOT.exists():
            subprocess.run(["git", "clone", REPO_URL, str(COLAB_PROJECT_ROOT)], check=True)
        os.chdir(COLAB_PROJECT_ROOT)
        current_dir = COLAB_PROJECT_ROOT.resolve()
    else:
        raise RuntimeError(
            "Hãy mở notebook từ project root hoặc cd vào thư mục truck_blind_spot trước khi chạy."
        )

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if IS_COLAB and (PROJECT_ROOT / "requirements.txt").exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt"), "--quiet"],
        check=True,
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")


## 1. Import và kiểm tra thiết bị

YOLOv9 dùng `device="0"` cho CUDA GPU đầu tiên hoặc `device="cpu"` khi không có GPU.


In [ ]:
import time
from pathlib import Path

import cv2
import torch
from IPython.display import Image, Video, clear_output, display

from src.pipeline import BlindSpotPipeline

device = "0" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Cấu hình demo

Các path đều resolve từ `PROJECT_ROOT`. Nếu muốn thử video hoặc weights khác, chỉ cần đổi các biến trong cell này.


In [ ]:
weights_path = "weights/best_roiv2.pt"
video_path = "assets/videos/demo4.mp4"
image_path = "assets/test.jpg"
roi_config_path = "configs/roi.json"
roi_profile = "front_camera"
classes_config_path = "configs/classes.yaml"

conf_threshold = 0.25
iou_threshold = 0.45
prediction_horizon_s = 1.0
alert_threshold = 0.6

output_video_path = PROJECT_ROOT / "outputs" / "demo_notebook_output.mp4"
max_preview_frames = 120


def require_file(relative_path: str) -> Path:
    resolved = PROJECT_ROOT / relative_path
    if not resolved.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {resolved}")
    return resolved


require_file(weights_path)
require_file(video_path)
require_file(roi_config_path)
require_file(classes_config_path)

print("Cấu hình sẵn sàng")
print(f"Weights: {weights_path}")
print(f"Video: {video_path}")
print(f"ROI profile: {roi_profile}")


## 3. Khởi tạo pipeline

`process_frame()` của pipeline hiện trả về `(annotated_frame, detections, tracks)`. Motion prediction mới nhất được lưu trong `pipeline.last_predictions`.


In [ ]:
pipeline = BlindSpotPipeline(
    weights_path=weights_path,
    roi_config_path=roi_config_path,
    roi_profile=roi_profile,
    classes_config_path=classes_config_path,
    device=device,
    conf_threshold=conf_threshold,
    iou_threshold=iou_threshold,
    prediction_horizons_s=[prediction_horizon_s],
    alert_confidence_threshold=alert_threshold,
)

print("Pipeline đã khởi tạo")


## 4. Chạy thử trên một ảnh

Cell này hữu ích để kiểm tra model, ROI và visualizer trước khi chạy video dài.


In [ ]:
sample_image = PROJECT_ROOT / image_path
if sample_image.exists():
    frame = cv2.imread(str(sample_image))
    if frame is None:
        raise RuntimeError(f"Không đọc được ảnh: {sample_image}")

    annotated_frame, detections, tracks = pipeline.process_frame(frame)
    success, buffer = cv2.imencode(".jpg", annotated_frame)
    if not success:
        raise RuntimeError("Không encode được ảnh kết quả.")

    display(Image(data=buffer.tobytes()))
    print(f"Detections: {len(detections)} | Tracks: {len(tracks)}")
else:
    print(f"Bỏ qua ảnh mẫu vì không tìm thấy: {sample_image}")


## 5. Chạy video demo trong notebook

Cell này xử lý tối đa `max_preview_frames` frame để notebook phản hồi nhanh. File kết quả được lưu vào `outputs/demo_notebook_output.mp4`.


In [ ]:
video_file = require_file(video_path)
cap = cv2.VideoCapture(str(video_file))
if not cap.isOpened():
    raise RuntimeError(f"Không thể mở video: {video_file}")

fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
output_video_path.parent.mkdir(parents=True, exist_ok=True)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(output_video_path), fourcc, fps, (width, height))
if not writer.isOpened():
    cap.release()
    raise RuntimeError(f"Không thể tạo output video: {output_video_path}")

frame_count = 0
last_frame = None
started_at = time.perf_counter()

try:
    while frame_count < max_preview_frames:
        success, frame = cap.read()
        if not success:
            break

        annotated_frame, detections, tracks = pipeline.process_frame(frame)
        writer.write(annotated_frame)
        last_frame = annotated_frame
        frame_count += 1

        if frame_count == 1 or frame_count % 15 == 0:
            clear_output(wait=True)
            elapsed = time.perf_counter() - started_at
            current_fps = frame_count / max(elapsed, 1e-6)
            print(
                f"Frame {frame_count}/{max_preview_frames} | "
                f"detections={len(detections)} | tracks={len(tracks)} | fps={current_fps:.1f}"
            )
            success_enc, buffer = cv2.imencode(".jpg", annotated_frame)
            if success_enc:
                display(Image(data=buffer.tobytes()))
finally:
    cap.release()
    writer.release()

elapsed = time.perf_counter() - started_at
print(f"Đã xử lý {frame_count} frame trong {elapsed:.1f}s")
print(f"Video output: {output_video_path}")

if last_frame is not None:
    success_enc, buffer = cv2.imencode(".jpg", last_frame)
    if success_enc:
        display(Image(data=buffer.tobytes()))

display(Video(str(output_video_path), embed=True))


## 6. Xem cảnh báo motion prediction mới nhất

Cảnh báo medium/high được lấy từ `pipeline.last_predictions`, cùng logic với `app.py`.


In [ ]:
predictions = pipeline.last_predictions or {}
active_alerts = [
    (track_id, pred.alert_level, pred.overall_confidence)
    for track_id, pred in predictions.items()
    if pred.alert_level in {"medium", "high"}
]

if active_alerts:
    for track_id, alert_level, confidence in active_alerts:
        print(f"Track {track_id}: {alert_level.upper()} | confidence={confidence:.2f}")
else:
    print("Không có cảnh báo medium/high ở frame cuối.")


## 7. Lệnh CLI tương đương

Khi cần chạy realtime toàn bộ video bằng cửa sổ OpenCV, dùng:

```bash
python app.py \
  --source assets/videos/demo4.mp4 \
  --weights weights/best_roiv2.pt \
  --roi configs/roi.json \
  --roi-profile front_camera \
  --classes-config configs/classes.yaml \
  --prediction-horizon 1.0 \
  --alert-threshold 0.6
```
